#Cleanup all endpoints and indexes
Run this notebook only if you want to remove all endpoints and indexes

In [0]:
%pip install -U -qq databricks-vectorsearch databricks-sdk 
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
import time

client = VectorSearchClient(disable_notice=True)
w = WorkspaceClient()

INDEX_DELETE_WAIT_SECS = 30
ENDPOINT_DELETE_WAIT_SECS = 20
MAX_POLLS = 40

def safe_list_endpoints():
    return client.list_endpoints().get("endpoints", [])

def safe_list_indexes(endpoint_name):
    return client.list_indexes(endpoint_name).get("vector_indexes", [])

# -------------------------------------------------------------------
# Step 1: Delete every index from every endpoint
# -------------------------------------------------------------------
print("Step 1: Deleting all indexes...")

endpoints = safe_list_endpoints()
if not endpoints:
    print("No endpoints found.")
else:
    print(f"Found {len(endpoints)} endpoint(s): {[e['name'] for e in endpoints]}")

all_index_names = []

for ep in endpoints:
    ep_name = ep["name"]
    indexes = safe_list_indexes(ep_name)
    print(f"\nEndpoint: {ep_name} | indexes: {len(indexes)}")

    for idx in indexes:
        idx_name = idx["name"]
        all_index_names.append(idx_name)
        print(f"  Deleting index: {idx_name}")
        try:
            # Use WorkspaceClient - bypasses endpoint association check
            w.vector_search_indexes.delete_index(index_name=idx_name)
            print(f"  ✓ Delete requested")
        except Exception as e:
            print(f"  ✗ Failed: {e}")

# -------------------------------------------------------------------
# Step 2: Poll until all indexes are gone
# -------------------------------------------------------------------
if all_index_names:
    print("\nStep 2: Waiting for index deletion to propagate...")
    remaining = set(all_index_names)

    for poll in range(MAX_POLLS):
        still_there = set()

        for idx_name in list(remaining):
            try:
                # Use WorkspaceClient for consistent get_index behavior
                w.vector_search_indexes.get_index(index_name=idx_name)
                still_there.add(idx_name)
            except Exception:
                pass  # Exception means index is gone

        if not still_there:
            print("✓ All indexes are gone")
            break

        print(f"[poll {poll+1}] still present: {sorted(still_there)}")
        remaining = still_there
        time.sleep(INDEX_DELETE_WAIT_SECS)
    else:
        print("✗ Timed out waiting for index deletion")
else:
    print("\nNo indexes found to delete.")

# -------------------------------------------------------------------
# Step 3: Delete all endpoints
# -------------------------------------------------------------------
print("\nStep 3: Deleting all endpoints...")
endpoints = safe_list_endpoints()
deleted_endpoints = []

for ep in endpoints:
    ep_name = ep["name"]
    try:
        client.delete_endpoint(ep_name)
        deleted_endpoints.append(ep_name)
        print(f"  ✓ Deleted endpoint: {ep_name}")
    except Exception as e:
        print(f"  ✗ Failed to delete {ep_name}: {e}")

# -------------------------------------------------------------------
# Step 4: Poll until all endpoints are gone
# -------------------------------------------------------------------
if deleted_endpoints:
    print("\nStep 4: Waiting for endpoint deletion to propagate...")
    remaining = set(deleted_endpoints)

    for poll in range(MAX_POLLS):
        current_names = {e["name"] for e in safe_list_endpoints()}
        still_there = remaining.intersection(current_names)

        if not still_there:
            print("✓ All endpoints are gone")
            break

        print(f"[poll {poll+1}] still present: {sorted(still_there)}")
        remaining = still_there
        time.sleep(ENDPOINT_DELETE_WAIT_SECS)
    else:
        print("✗ Timed out waiting for endpoint deletion")
else:
    print("\nNo endpoints to delete.")

print("\n✓ Cleanup complete. Safe to create a fresh endpoint and index.")